# Step 1 — Extração da série temporal Sentinel-2

Extrai, via Google Earth Engine, um GeoTIFF por ano num buffer ao redor de um ponto (lat/lon) e gera composições RGB em JPG para inspeção visual.

A lógica reutilizável (`mask_s2_clouds`, `extract_datacenter_timeseries`, `export_rgb_jpgs`) vive em [`src/extraction.py`](../src/extraction.py) — este notebook só chama essas funções. Para rodar via linha de comando, veja `scripts/step1_extracao_imagens_satelite.py`.

In [ ]:
import os
import sys
from pathlib import Path

# Garante que a raiz do repositório está no sys.path e é o cwd, independente de onde o
# Jupyter foi iniciado — assim `from src... import ...` e os caminhos relativos
# ('data/raw', 'imagens_jpg', ...) funcionam sempre.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'requirements.txt').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import ee
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from src.extraction import RAW_DIR, export_rgb_jpgs, extract_datacenter_timeseries, tif_to_rgb

load_dotenv()  # carrega variáveis de ambiente do arquivo .env, se existir

In [ ]:
# Autenticação e inicialização do Earth Engine
# ee.Authenticate()  # descomente e rode na primeira vez (colar no terminal)

EE_PROJECT = os.environ.get('EE_PROJECT')
if not EE_PROJECT:
    raise RuntimeError(
        "Defina a variável de ambiente EE_PROJECT com o ID do seu projeto no Google Cloud "
        "antes de rodar esta célula (ex.: PowerShell: $env:EE_PROJECT = 'seu-projeto-id'; "
        "ou crie um arquivo .env com EE_PROJECT=seu-projeto-id)."
    )

ee.Initialize(project=EE_PROJECT)

In [ ]:
name_datacenter = 'Ascenty_Vinhedo'
lat = -23.071035
lon = -47.011837
year_list = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

extract_datacenter_timeseries(name_datacenter, lat, lon, year_list, out_dir=RAW_DIR)

In [ ]:
path = os.path.join(RAW_DIR, 'Ascenty_Vinhedo_2024.tif')

rgb = tif_to_rgb(path)

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title('Ascenty Vinhedo 2024')
plt.axis('off')
plt.show()

## Gera as composições RGB (JPG) de toda a série temporal

In [ ]:
export_rgb_jpgs(pasta_entrada=RAW_DIR, pasta_saida='imagens_jpg')